# `gptsovits` with VoiceHub

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/models/gptsovits.ipynb)

- Task: **Text to speech**
- Hugging Face ID: [`lj1995/GPT-SoVITS`](https://huggingface.co/lj1995/GPT-SoVITS)

Install VoiceHub using the [installation guide](https://kadirnar.github.io/voicehub/getting-started/installation/)
before opening this model workflow. This notebook contains no package-install cell.

The registry check is safe to run without downloading weights. Inference is disabled by default.


In [ ]:
from pathlib import Path

RUN_INFERENCE = False
MODEL_TYPE = 'gptsovits'
CHECKPOINT = 'lj1995/GPT-SoVITS'
DEVICE = "cuda"
TEXT = 'VoiceHub keeps model integrations explicit and reproducible.'
OUTPUT_FILE = Path("artifacts/gptsovits.wav")


## Inspect registry support


In [ ]:
from voicehub import get_model_spec

model_spec = get_model_spec(MODEL_TYPE)
assert model_spec.task.value == 'text-to-speech'
assert model_spec.default_model_path == CHECKPOINT
print("task:", model_spec.task.value)
print("checkpoint:", model_spec.default_model_path)
print("capabilities:", ", ".join(model_spec.capabilities))
print("training:", model_spec.training.support.value)


## Run inference

Defines both target and prompt languages for GPT-SoVITS zero-shot voice prompting.

Use the language codes accepted by the selected GPT-SoVITS checkpoint and an exact prompt transcript.

This VoiceHub example is maintained in this repository and is not copied from an upstream package snippet. Set `RUN_INFERENCE = True` after reviewing inputs.


In [ ]:
if RUN_INFERENCE:
    from voicehub import AutoModelForTextToSpeech, TTSGenerationConfig

    REFERENCE_AUDIO = Path("reference.wav")
    REFERENCE_TEXT = "The reference transcript must exactly match the authorized audio."
    if not REFERENCE_AUDIO.is_file():
        raise FileNotFoundError(REFERENCE_AUDIO)

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    model = AutoModelForTextToSpeech.from_pretrained(
        CHECKPOINT,
        model_type=MODEL_TYPE,
        device=DEVICE,
        lazy_load=True,
    )
    output = model.generate(
        TEXT,
        generation_config=TTSGenerationConfig(seed=42, output_file=OUTPUT_FILE),
        text_language="en",
        speaker_audio_path=str(REFERENCE_AUDIO),
        prompt_language="en",
        prompt_text=REFERENCE_TEXT,
        text_split_method="cut5",
    )
    print(output.file_path, output.sample_rate, output.metadata)


## Next

See the [inference guide](https://kadirnar.github.io/voicehub/guides/inference/) and [model catalog](https://kadirnar.github.io/voicehub/models/) for the shared runtime contract and model-specific limitations.
